# M8_8.31–M8_8.33 · Funciones, automatización y reproducibilidad

Este bloque transforma operaciones ya conocidas en un flujo reutilizable. No repite pandas ni estadística desde cero.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_NAME = "m8-herramientas-data-science"
GITHUB_USER = "REPLACE_WITH_YOUR_GITHUB_USERNAME"  # El profesorado lo cambia una vez antes de publicar

def localizar_repo():
    actual = Path.cwd().resolve()
    for candidato in [actual, *actual.parents]:
        if (candidato / "data" / "input").exists():
            return candidato
    if "google.colab" in sys.modules:
        destino = Path("/content") / REPO_NAME
        if not destino.exists():
            url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
            if GITHUB_USER.startswith("REPLACE_"):
                raise RuntimeError("El profesorado debe configurar GITHUB_USER antes de publicar el repositorio.")
            subprocess.run(["git", "clone", "--depth", "1", url, str(destino)], check=True)
        return destino
    raise FileNotFoundError("No se encuentra la raiz del repositorio.")

ROOT = localizar_repo()
DATA = ROOT / "data" / "input"
DB = ROOT / "data" / "database" / "hidrogeologia.sqlite"
OUTPUT = ROOT / "data" / "output"
for carpeta in [OUTPUT / "tables", OUTPUT / "figures", OUTPUT / "logs"]:
    carpeta.mkdir(parents=True, exist_ok=True)
print("Repositorio:", ROOT)

# M8_8.31 · Funciones y organización

Una función debe tener una responsabilidad clara, entradas explícitas y un resultado reutilizable. La validación se realiza antes del cálculo.

In [ ]:
import pandas as pd, numpy as np, sqlite3, matplotlib.pyplot as plt, json, datetime, sys
with sqlite3.connect(DB) as con:
    datos=pd.read_sql_query("""SELECT m.*,p.acuifero,p.cota_terreno_m FROM mediciones m JOIN pozos p ON m.id_pozo=p.id_pozo""",con,parse_dates=["fecha"])
display(datos.head())

In [ ]:
def seleccionar_pozo(datos,id_pozo):
    if id_pozo not in set(datos.id_pozo):
        raise ValueError(f"Pozo desconocido: {id_pozo}")
    return datos.loc[datos.id_pozo==id_pozo].copy()

def validar(datos):
    req={"id_pozo","fecha","profundidad_nivel_m"}; faltan=req-set(datos.columns)
    if faltan: raise ValueError(f"Faltan columnas: {faltan}")
    return {"filas":len(datos),"ausentes":int(datos.profundidad_nivel_m.isna().sum()),"duplicados":int(datos.duplicated().sum())}

def resumir(datos,variable):
    s=pd.to_numeric(datos[variable],errors="coerce")
    return pd.Series({"n":s.count(),"media":s.mean(),"mediana":s.median(),"min":s.min(),"max":s.max()})
validar(datos)

## Scripts, módulos y `main`

Las funciones pueden guardarse en `src`. Un script principal coordina carga, validación, análisis y salida. `if __name__ == "__main__":` evita ejecutar la parte principal al importar el módulo.

# M8_8.32 · Automatización

Automatizar significa aplicar el mismo procedimiento validado a varios pozos, archivos o escenarios, con parámetros y salidas trazables.

In [ ]:
resultados=[]
for pid in sorted(datos.id_pozo.unique()):
    r=resumir(seleccionar_pozo(datos,pid),"profundidad_nivel_m"); r["id_pozo"]=pid; resultados.append(r)
tabla=pd.DataFrame(resultados).set_index("id_pozo")
display(tabla)

## Configuración, rutas y errores

Los parámetros se separan de la lógica. `pathlib` evita rutas rígidas. `try/except` gestiona errores esperados y no debe silenciar fallos.

In [ ]:
config={"variable":"profundidad_nivel_m","pozos":["P01","P02","P03"],"dpi":150}
(OUTPUT/"logs/config.json").write_text(json.dumps(config,indent=2),encoding="utf-8")
for pid in config["pozos"]+["P99"]:
    try: print(pid,len(seleccionar_pozo(datos,pid)))
    except ValueError as e: print("ERROR",e)

# M8_8.33 · Flujo reproducible

El flujo completo carga, valida, analiza, guarda resultados y registra parámetros. Que termine sin error no garantiza validez científica.

In [ ]:
def guardar_figura(sub,ruta,dpi=150):
    fig,ax=plt.subplots(figsize=(7,3.5)); ax.plot(sub.fecha,sub.profundidad_nivel_m,marker="o"); ax.invert_yaxis()
    ax.set(xlabel="Fecha",ylabel="Profundidad (m)",title=f"Pozo {sub.id_pozo.iloc[0]}")
    fig.savefig(ruta,dpi=dpi,bbox_inches="tight"); plt.close(fig)

def ejecutar(datos,config):
    control=validar(datos); out=[]
    for pid in config["pozos"]:
        sub=seleccionar_pozo(datos,pid); r=resumir(sub,config["variable"]); r["id_pozo"]=pid; out.append(r)
        guardar_figura(sub,OUTPUT/f"figures/{pid}_nivel.png",config["dpi"])
    tabla=pd.DataFrame(out).set_index("id_pozo"); tabla.to_csv(OUTPUT/"tables/resumen_pozos.csv")
    registro={"fecha_utc":datetime.datetime.now(datetime.timezone.utc).isoformat(),"python":sys.version.split()[0],"config":config,"control":control}
    (OUTPUT/"logs/registro.json").write_text(json.dumps(registro,indent=2),encoding="utf-8")
    return tabla,control

tabla,control=ejecutar(datos,config); display(tabla); print(control)

## Comprobación final

- originales preservados;
- rutas relativas;
- columnas y unidades documentadas;
- parámetros explícitos;
- resultados trazables;
- errores visibles;
- instrucciones de ejecución;
- interpretación y limitación.

## Actividad

Adapta el flujo a otra selección de pozos o variable. Entrega tabla, figuras, configuración, registro y una interpretación breve con una limitación.